# 01 — RAG Exploration: Embedding Similarity Search Intuition

**Goal**: build intuition for how embedding-based retrieval works inside
sky-finance's strategy engine.

By the end of this notebook you will understand:
- What an embedding vector looks like and why cosine similarity works
- Why plain top-k is blind to minority-sentiment signals (e.g. the few
  negative articles in a bull run)
- How **sentiment-bucketed retrieval** solves this
- How the similarity threshold trades off precision against recall

**Prerequisites** — all must be running before executing cells:
```bash
# 1. PostgreSQL + pgvector
docker compose -f docker/docker-compose.yml up -d

# 2. Embedding model
ollama pull nomic-embed-text

# 3. Some data in the DB (at least one ingest + pipeline cycle)
uv run honcho start          # starts worker + beat; or trigger tasks manually
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sky_finance.storage.db import get_connection, open_pool
from sky_finance.storage.embedder import (
    embed_single, BACKEND, OLLAMA_MODEL, EMBEDDING_DIM,
)

%matplotlib inline

# --- connectivity checks ---------------------------------------------------
try:
    open_pool()
    with get_connection() as conn:
        conn.execute('SELECT 1')
    print('✓  DB connected')
except Exception as exc:
    print(f'✗  DB: {exc}')
    print('   → docker compose -f docker/docker-compose.yml up -d')

print(f'✓  Embedding backend : {BACKEND}')
print(f'   model             : {OLLAMA_MODEL}')
print(f'   dimensions        : {EMBEDDING_DIM}')

## 1. What does an embedding look like?

`nomic-embed-text` converts any piece of text into a **768-dimensional
L2-normalised vector**.  Because the vectors are unit-length, the dot product
equals the cosine similarity — so `<=>` in pgvector is a single cheap
operation with no normalisation overhead.

In [ ]:
TICKER = 'AAPL'    # ← change to any ticker you have in the DB
QUERY  = 'Apple iPhone supply chain disruption China'

vec  = embed_single(QUERY)
norm = np.linalg.norm(vec)

print(f'Query      : {QUERY!r}')
print(f'Shape      : ({len(vec)},)')
print(f'L2 norm    : {norm:.4f}  (should be ≈ 1.0 for nomic-embed-text)')
print(f'Min / Max  : {min(vec):.4f} / {max(vec):.4f}')
print(f'First 8 dims : {[round(v, 4) for v in vec[:8]]}')

## 2. Plain cosine similarity search

A naïve top-k query returns the documents *closest to the query vector*
regardless of their sentiment.

**The problem**: for a stock in a bull run the corpus may be 80 % positive.
Plain top-10 fills the LLM's context window with bullish articles and buries
the handful of negative signals that matter most for downside analysis.

In [ ]:
PLAIN_SQL = '''
    SELECT  d.title,
            d.sentiment,
            LEFT(d.body, 200),
            ROUND((1 - (e.embedding <=> %s::vector))::numeric, 3) AS sim
    FROM    embeddings e
    JOIN    documents  d ON d.id = e.document_id
    WHERE   e.ticker = %s
    ORDER   BY e.embedding <=> %s::vector
    LIMIT   10
'''

with get_connection() as conn:
    from pgvector.psycopg import register_vector
    register_vector(conn)
    with conn.cursor() as cur:
        cur.execute(PLAIN_SQL, (vec, TICKER, vec))
        plain_rows = cur.fetchall()

if not plain_rows:
    print('No rows — run ingest + pipeline first (uv run honcho start).')
else:
    print(f'Top 10 plain results  ticker={TICKER!r}  query={QUERY!r}\n')
    print(f'{"#":>2}  {"sim":>5}  {"sentiment":9}  title')
    print('-' * 80)
    for i, (title, sentiment, body, sim) in enumerate(plain_rows, 1):
        print(f'{i:2}  {float(sim):>5.3f}  {(sentiment or "?"):9}  {title[:55]}')

## 3. Why sentiment bucketing matters

sky-finance runs **three separate top-k queries** — one per sentiment bucket
— then merges and re-ranks by similarity.  This guarantees that negative
signals always reach the model, even when they represent only 5 % of the
corpus.

Each bucket's `top_k` is independently configurable per strategy:
```toml
rag_top_k_positive = 20
rag_top_k_neutral  = 20
rag_top_k_negative = 20   # ← always surfaced, even when rare
```

In [ ]:
BUCKET_SQL = '''
    SELECT  d.title,
            LEFT(d.body, 200),
            ROUND((1 - (e.embedding <=> %s::vector))::numeric, 3) AS sim
    FROM    embeddings e
    JOIN    documents  d ON d.id = e.document_id
    WHERE   e.ticker    = %s
      AND   d.sentiment = %s
      AND   1 - (e.embedding <=> %s::vector) >= %s
    ORDER   BY sim DESC
    LIMIT   %s
'''

THRESHOLD = 0.55
TOP_K     = 5
ICONS     = {'positive': '✅', 'neutral': '⬜', 'negative': '🔴'}

buckets: dict[str, list] = {}
with get_connection() as conn:
    from pgvector.psycopg import register_vector
    register_vector(conn)
    with conn.cursor() as cur:
        for sentiment in ('positive', 'neutral', 'negative'):
            cur.execute(BUCKET_SQL, (vec, TICKER, sentiment, vec, THRESHOLD, TOP_K))
            buckets[sentiment] = cur.fetchall()

for sentiment, rows in buckets.items():
    print(f'\n{ICONS[sentiment]}  {sentiment.upper()}  ({len(rows)} chunks  threshold={THRESHOLD})')
    for title, body, sim in rows:
        print(f'    {float(sim):.3f}  {title[:65]}')

## 4. Visualising similarity scores by sentiment

In [ ]:
all_rows = [
    (title, sentiment, float(sim))
    for sentiment, rows in buckets.items()
    for title, _, sim in rows
]
all_rows.sort(key=lambda x: x[2], reverse=True)

COLORS = {'positive': '#22c55e', 'neutral': '#94a3b8', 'negative': '#ef4444'}

if not all_rows:
    print('No data to plot — run ingest + pipeline first.')
else:
    bar_colors = [COLORS[r[1]] for r in all_rows]
    labels     = [f'[{r[1][:3].upper()}] {r[0][:50]}' for r in all_rows]
    sims       = [r[2] for r in all_rows]

    fig, ax = plt.subplots(figsize=(13, max(4, len(all_rows) * 0.5)))
    ax.barh(range(len(all_rows)), sims, color=bar_colors, edgecolor='white', height=0.7)
    ax.set_yticks(range(len(all_rows)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Cosine Similarity', fontsize=11)
    ax.set_xlim(0, 1.05)
    ax.set_title(f'Sentiment-bucketed RAG chunks for {TICKER}\n"{QUERY}"', fontsize=12)
    ax.axvline(x=THRESHOLD, color='black', linestyle='--', linewidth=1, alpha=0.6)
    ax.text(THRESHOLD + 0.01, -0.7, f'threshold={THRESHOLD}', fontsize=8, alpha=0.7)
    patches = [mpatches.Patch(color=COLORS[s], label=s) for s in COLORS]
    ax.legend(handles=patches, loc='lower right', fontsize=9)
    fig.tight_layout()
    plt.show()

## 5. Threshold: precision vs recall

The threshold drops any chunk whose similarity falls below the cutoff.

- **Lower threshold** → more recall, noisier context (off-topic macro articles)
- **Higher threshold** → more precision, risks missing relevant but
  differently-worded articles

**Production default** `0.55` was calibrated empirically on nomic-embed-text
768-dim vectors against financial news.

In [ ]:
with get_connection() as conn:
    from pgvector.psycopg import register_vector
    register_vector(conn)
    with conn.cursor() as cur:
        print(f'{"Threshold":>10}  {"positive":>9}  {"neutral":>8}  {"negative":>9}  {"total":>6}')
        print('-' * 55)
        for thresh in [0.30, 0.45, 0.55, 0.65, 0.75, 0.85]:
            counts = {}
            for sentiment in ('positive', 'neutral', 'negative'):
                cur.execute(
                    '''
                    SELECT COUNT(*) FROM embeddings e
                    JOIN documents d ON d.id = e.document_id
                    WHERE e.ticker = %s AND d.sentiment = %s
                      AND 1 - (e.embedding <=> %s::vector) >= %s
                    ''',
                    (TICKER, sentiment, vec, thresh),
                )
                counts[sentiment] = cur.fetchone()[0]
            total  = sum(counts.values())
            marker = '  ← production default' if thresh == 0.55 else ''
            print(
                f'{thresh:>10.2f}  {counts["positive"]:>9}  {counts["neutral"]:>8}'
                f'  {counts["negative"]:>9}  {total:>6}{marker}'
            )

## Key Takeaways

1. **Embeddings are L2-normalised dense vectors** — cosine similarity is a
   cheap dot product, and `<=>` in pgvector executes it with an ANN index.

2. **Plain top-k favours the majority sentiment** — in a bull-run corpus,
   plain retrieval fills the LLM's context with bullish noise.

3. **Bucketed retrieval is the core RAG insight** — three separate queries
   guarantee all three sentiment signals reach the model, even when negative
   news is a tiny minority of the corpus.

4. **Threshold 0.55 is a calibrated choice** — below this value retrieved
   chunks are off-topic macro articles with incidental ticker mentions;
   above 0.75 you lose recall on relevant but differently-worded articles.

5. **Experiment** — change `TICKER` and `QUERY`, re-run the cells, and watch
   how the similarity distribution shifts.  This is exactly the intuition the
   strategy engine relies on.